# **MeetingMind AI Agent: A Retrieval-Augmented Meeting Assistant**

**An Intelligent Meeting Assistant Powered by Generative AI and Retrieval-Augmented Generation (RAG)**

**Capstone Project**


Project Type: Single-Agent AI Application

**Project Overview**

MeetingMind AI Agent is an intelligent meeting assistant that helps users analyze meeting transcripts and notes using a Large Language Model (LLM). The application automatically generates structured outputs such as executive summaries, key decisions, action items, follow-up emails, and suggested agendas for future meetings.

The system also allows users to ask questions about uploaded meeting documents through Retrieval-Augmented Generation (RAG), ensuring responses are based on the meeting content rather than general knowledge.

**Problem Statement**

Meetings often generate lengthy notes and transcripts that take time to review. Identifying important decisions, assigned responsibilities, deadlines, and follow-up actions can be tedious and may result in important information being overlooked.

**Proposed Solution**

MeetingMind AI Agent simplifies meeting management by using Artificial Intelligence to analyze meeting transcripts and generate accurate, structured, and actionable outputs. The application combines a Large Language Model with Retrieval-Augmented Generation (RAG) to answer questions based on uploaded meeting documents.

**Project Objectives**

Build a single-agent AI application.
Integrate a Large Language Model (LLM).
Allow users to upload meeting transcripts.
Implement Retrieval-Augmented Generation (RAG).
Generate meeting summaries and action items.
Produce follow up emails and suggested next meeting agendas.
Deploy the application using Streamlit.

**Technologies Used**

Python

Google Colab

LangChain

Chroma Vector Database (ChromaDB)

Sentence Transformers

Streamlit

GitHub

### **Install Required Libraries**

In [ ]:
# Install required libraries

!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q chromadb
!pip install -q pypdf
!pip install -q langchain-text-splitters

print("All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GOOGLE_API_KEY") # Ensure API key is loaded

client = genai.Client(api_key=GEMINI_API_KEY)

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

**Step 2  Importing Libraries and Initializing the Gemini Model**

After installing the required packages in the previous step, the next task is to import the libraries that will be used throughout the project. Each library has a specific responsibility in the Retrieval-Augmented Generation (RAG) pipeline.

The project begins by importing Python's built-in os module, which provides access to operating system functions when needed. The userdata and files modules from Google Colab are also imported. The userdata module is used to securely retrieve the Gemini API key without exposing it in the notebook, while the files module allows users to upload meeting transcript PDFs directly into the Colab environment.

Several components from the LangChain framework are then imported to simplify document processing. PyPDFLoader is responsible for reading the uploaded PDF document page by page, making it possible to extract the meeting transcript into text format. Although TextLoader is also imported to support plain text files, this project focuses primarily on PDF meeting transcripts.

In [ ]:
import os
from google.colab import userdata, files

# LangChain components
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceInstructEmbeddings
from langchain_community.vectorstores import Chroma

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# Load Gemini API key
GEMINI_API_KEY = userdata.get("GOOGLE_API_KEY") # Corrected to use the secret name

# Create Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", # Updated to a newer, available model
    google_api_key=GEMINI_API_KEY,
    temperature=0
)

print("Gemini API loaded successfully!")

/tmp/ipykernel_3466/1074889950.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Gemini API loaded successfully!


 **Step 3  Uploading the Meeting Transcript**

Upload a meeting transcript in PDF format.

Recommended documents:

- Meeting minutes
- Project meeting transcript
- Team discussion notes
- Business meeting report

In [ ]:
# Upload your meeting transcript (PDF)

print("Upload a meeting transcript (PDF)...")

uploaded = files.upload()

if uploaded:
    doc_filename = list(uploaded.keys())[0]
    print(f"\nUploaded: {doc_filename}")
else:
    print("No file uploaded.")

Upload a meeting transcript (PDF)...


Saving Boardroom_Company_Staff_Meeting_Transcript.pdf to Boardroom_Company_Staff_Meeting_Transcript.pdf

Uploaded: Boardroom_Company_Staff_Meeting_Transcript.pdf


**Step 4  Loading the Uploaded PDF Document**

After uploading the meeting transcript, the next step is to read its contents. The PyPDFLoader is used to extract the text from each page of the PDF and store it as a document object. The program also displays the total number of pages loaded and shows a preview of the first page to confirm that the document has been read successfully.

In [ ]:
# Load the uploaded meeting transcript

loader = PyPDFLoader(doc_filename)

documents = loader.load()

print(f"Number of pages loaded: {len(documents)}")

# Display the first page
print("\nFirst Page Preview:\n")
print(documents[0].page_content[:1000])




Number of pages loaded: 1

First Page Preview:

BOARDROOM COMPANY STAFF MEETING
 TRANSCRIPT
Company: Apex Innovations Ltd.
Date: 15 July 2026
Time: 9:00 AM – 10:30 AM
Venue: Executive Boardroom
Chairperson: Mrs. Grace Okafor (Managing Director)
Attendees: Grace Okafor, Daniel Musa, Sarah James, Michael Adeyemi, Linda Eze, Peter
Johnson, Esther Bello.
Agenda
Review previous minutes; Quarterly sales; Budget; CRM upgrade; Customer feedback; Staff
training; Action items.
Discussion
The Managing Director welcomed everyone and reviewed progress since the last meeting. Sales
increased by 18% over the previous quarter. Finance approved additional funding for digital
transformation. The IT Manager confirmed that the new CRM system is 85% complete and security
testing will finish next week. HR proposed a two-day staff training programme covering customer
service, cybersecurity, and communication. Customer Service recommended an automated
ticketing system to improve response times.
Decisions
Appr

**Step 5 Splitting the Document into Text Chunks**

Large documents are difficult for language models to process at once, so the meeting transcript is divided into smaller text chunks. A chunk size of 800 characters with an overlap of 100 characters is used to preserve context between consecutive chunks. This improves the accuracy of information retrieval during the question-answering process. The first chunk is displayed as a preview to verify that the splitting was successful.

In [ ]:
# Split the meeting transcript into smaller chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

# Preview the first chunk
print("\nFirst Chunk Preview:\n")
print(chunks[0].page_content)

Total chunks created: 2

First Chunk Preview:

BOARDROOM COMPANY STAFF MEETING
 TRANSCRIPT
Company: Apex Innovations Ltd.
Date: 15 July 2026
Time: 9:00 AM – 10:30 AM
Venue: Executive Boardroom
Chairperson: Mrs. Grace Okafor (Managing Director)
Attendees: Grace Okafor, Daniel Musa, Sarah James, Michael Adeyemi, Linda Eze, Peter
Johnson, Esther Bello.
Agenda
Review previous minutes; Quarterly sales; Budget; CRM upgrade; Customer feedback; Staff
training; Action items.
Discussion
The Managing Director welcomed everyone and reviewed progress since the last meeting. Sales
increased by 18% over the previous quarter. Finance approved additional funding for digital
transformation. The IT Manager confirmed that the new CRM system is 85% complete and security


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Import Successful!")

Import Successful!


**Step 6 Creating Embeddings and Building the Vector Database**

 In this step, each text chunk is converted into a numerical representation called an embedding using the all-MiniLM-L6-v2 model. These embeddings capture the meaning of the text, making it possible to search based on context rather than exact words. The embeddings are then stored in ChromaDB, which acts as the project's vector database. This allows the system to quickly retrieve the most relevant chunks when the user asks a question.

In [ ]:
print("Loading free local embedding model (all-MiniLM-L6-v2)...")
print("This downloads ~80MB the first time — takes about 30 seconds.")
print()

# Free local embedding model — no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
    # Fast, accurate, widely used for RAG demos
)

print("Embedding model loaded. Now embedding chunks and storing in ChromaDB...")

# Create ChromaDB from our chunks
# This embeds EACH chunk and stores the vector + original text
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_document"
)

print()
print(f"Vector store ready!")
print(f"  {len(chunks)} chunks embedded and stored")
print(f"  Each chunk is now a vector of numbers")
print(f"  ChromaDB can find similar chunks in milliseconds")

Loading free local embedding model (all-MiniLM-L6-v2)...
This downloads ~80MB the first time — takes about 30 seconds.



/tmp/ipykernel_3466/3356964342.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded. Now embedding chunks and storing in ChromaDB...

Vector store ready!
  2 chunks embedded and stored
  Each chunk is now a vector of numbers
  ChromaDB can find similar chunks in milliseconds


In [ ]:
print(vectorstore)

In [ ]:
print(llm)

metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.7'}} output_version=None profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') location=None model='gemini-3.5-flash' temperature=0.0 client=<google.genai.client.Client object at 0x7d7445057b90> default_metadata=() model_kwargs={}


In [15]:
import streamlit as st
import os
from datetime import datetime # Import datetime

# LangChain components
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings # Updated import
from langchain_community.vectorstores import Chroma

# Groq
from langchain_groq import ChatGroq

@st.cache_resource
def load_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

# Authentication and DB
from auth_db import init_db, register_user, verify_user, save_conversation, get_conversations

# Removed: from google.colab import userdata # Added import for userdata

# --- Streamlit UI Configuration ---
st.set_page_config(
    page_title="MeetingMind AI Agent",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.image("meetingmind_banner.png", width='stretch') # Changed use_container_width=True to width='stretch'

# Initialize the database
init_db()

# Session state for authentication
if 'authenticated' not in st.session_state:
    st.session_state.authenticated = False
if 'user_id' not in st.session_state:
    st.session_state.user_id = None
if 'username' not in st.session_state:
    st.session_state.username = None


def show_auth_page():
    """Displays the login/signup forms."""
    st.markdown("""
    Welcome! Please Login or Sign Up to use the MeetingMind AI Agent.
    """)

    tab1, tab2 = st.tabs(["Login", "Sign Up"])

    with tab1:
        st.subheader("Login") # Changed st.text back to st.subheader
        login_username = st.text_input("Username", key="login_username")
        login_password = st.text_input("Password", type="password", key="login_password")
        if st.button("Login"):
            user_id = verify_user(login_username, login_password)
            if user_id:
                st.session_state.authenticated = True
                st.session_state.user_id = user_id
                st.session_state.username = login_username
                st.rerun()
            else:
                st.error("Invalid username or password")

    with tab2:
        st.subheader("Sign Up")
        new_username = st.text_input("New Username", key="new_username")
        new_password = st.text_input("New Password", type="password", key="new_password")
        confirm_password = st.text_input("Confirm Password", type="password", key="confirm_password")
        if st.button("Register"):
            if new_password != confirm_password:
                st.error("Passwords do not match!")
            elif len(new_username) < 3 or len(new_password) < 6:
                st.error("Username must be at least 3 characters and password at least 6 characters long.")
            else:
                if register_user(new_username, new_password):
                    st.success("Registration successful! Please login.")
                else:
                    st.error("Username already exists or registration failed.")


def logout():
    st.session_state.authenticated = False
    st.session_state.user_id = None
    st.session_state.username = None
    st.session_state.vectorstore = None
    st.session_state.pdf_uploaded = False
    st.rerun()


if not st.session_state.authenticated:
    show_auth_page()
else:
    with st.sidebar:
        st.header(f"Welcome, {st.session_state.username}!")
        if st.button("Logout"):
            logout()
        st.markdown("--- ")
        st.header("📚 How to Use")
        with st.expander("Click here for detailed instructions", expanded=True):
            st.markdown("""
            1.  **Upload PDF**: Use the file uploader below to select your meeting transcript in PDF format.
            2.  **Ask Questions**: Once processed, type your questions about the meeting content into the text area.
            3.  **Get Answers**: Click 'Get Answer' to receive a concise, accurate response generated by Google Gemini, based *only* on your uploaded document.
            """)
        st.markdown("--- ")
        st.header("💡 About This App")
        st.info("""
        MeetingMind AI Agent helps you cut through lengthy meeting notes to find key information fast. It uses advanced AI to summarize, extract decisions, and answer questions directly from your documents.

        Developed as a capstone project utilizing LangChain, ChromaDB, HuggingFace Embeddings, and Google Gemini.
        """)

        st.markdown("--- ")
        st.header("Past Conversations")
        user_conversations = get_conversations(st.session_state.user_id)
        if user_conversations:
            for i, (timestamp, question, answer) in enumerate(user_conversations):
                with st.expander(f"**{datetime.fromisoformat(timestamp).strftime('%Y-%m-%d %H:%M')}**: {question[:50]}..."):
                    st.markdown(f"**Q:** {question}")
                    st.markdown(f"**A:** {answer}")
        else:
            st.info("No past conversations found.")

    if 'vectorstore' not in st.session_state:
        st.session_state.vectorstore = None
    if 'pdf_uploaded' not in st.session_state:
        st.session_state.pdf_uploaded = False

    # Groq API Key input (using st.secrets for security)
    groq_api_key = st.secrets.get("GROQ_API_KEY", "")
    with st.container(border=True):
        st.subheader("Upload Your Meeting Transcript (PDF)")
        uploaded_file = st.file_uploader("Choose a PDF file to analyze", type="pdf", label_visibility="collapsed")

        if uploaded_file is not None:
            st.session_state.pdf_uploaded = True
            st.success("PDF uploaded successfully! Now processing...")

            with open("uploaded_meeting.pdf", "wb") as f:
                f.write(uploaded_file.getbuffer())
            doc_filename = "uploaded_meeting.pdf"

            st.write("#### Document Processing Status")

            with st.spinner("Loading PDF..."):
                try:
                    loader = PyPDFLoader(doc_filename)
                    documents = loader.load()
                    st.success(f"Loaded {len(documents)} pages.")
                except Exception as e:
                    st.error(f"Error loading PDF: {e}")
                    st.error("This could be due to a corrupted PDF, a password-protected file, or an unsupported format.")
                    st.error("Please ensure the PDF is valid and not encrypted.")
                    st.stop() # Stop further execution if PDF loading fails

            with st.spinner("Splitting document into chunks..."):
                text_splitter = RecursiveCharacterTextSplitter(
                    chunk_size=1200,
                    chunk_overlap=200
                )
                chunks = text_splitter.split_documents(documents)
                st.success(f"Created {len(chunks)} chunks.")

            with st.spinner("Embedding chunks and creating vector store (this may take a moment)..."):
                embeddings = load_embeddings()

                st.session_state.vectorstore = Chroma.from_documents(
                    documents=chunks,
                    embedding=embeddings
                )

                st.success("Vector store created successfully!")
    if st.session_state.pdf_uploaded and st.session_state.vectorstore is not None:
        st.markdown("--- ")
        with st.container(border=True):
            st.subheader("Ask a Question About the Meeting")
            question_input = st.text_area("Enter your question here:", key="question_area", height=100)

    if st.button("Get Answer"):

        if not groq_api_key:
            st.error("Groq API Key not found in Streamlit Secrets.")

        else:
            llm = ChatGroq(
                groq_api_key=groq_api_key,
                model_name="llama-3.3-70b-versatile",
                temperature=0
            )

            def ask_document(question):
                retriever = st.session_state.vectorstore.as_retriever(
                    search_kwargs={"k": 8}
                )

                retrieved_chunks = retriever.invoke(question)

                unique_chunks_content = set()
                deduplicated_chunks = []
                for chunk in retrieved_chunks:
                    if chunk.page_content not in unique_chunks_content:
                        unique_chunks_content.add(chunk.page_content)
                        deduplicated_chunks.append(chunk)

                context = "\n\n---\n\n".join(
                    [chunk.page_content for chunk in deduplicated_chunks]
                )

                user_prompt = f"""You are an AI assistant designed to extract and summarize information from meeting transcripts.\nBased on the following CONTEXT from the meeting, please answer the QUESTION.\nYour answer should be as comprehensive and informative as possible, drawing all relevant details and making logical inferences *only* from the provided CONTEXT.\nIf the CONTEXT does not contain enough information to provide a direct answer, or if the information is entirely absent, explain what information is missing or state that the answer cannot be found within the document.\nDo not use any external knowledge.\n\nCONTEXT:\n{context}\n\nQUESTION:\n{question}\n"""

                try:
                    response = llm.invoke(user_prompt)
                    answer = str(response.content)

                except Exception as e:
                    st.error(f"Groq Error: {e}")
                    return (
                        "⚠️ Groq is temporarily unavailable.",
                        []
                    )

                return answer, deduplicated_chunks

            with st.spinner("Generating answer with Groq AI..."):
                answer, source_chunks = ask_document(question_input)

                st.success("Answer Generated!")
                st.markdown("#### Answer:")
                st.write(answer)

                # Save conversation
                save_conversation(st.session_state.user_id, question_input, answer)

                # Removed the st.expander that displayed source chunks
    else:
        st.info("Please upload a PDF document to begin asking questions.")

2026-07-15 15:03:45.046 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 15:03:45.047 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


RuntimeError: Runtime hasn't been created!

**Step 7  Test Document Retrieval**

In this step, I tested the Retrieval part of my RAG system before connecting it to Gemini. Instead of sending the entire meeting transcript to the AI model, ChromaDB searches the vector database and retrieves only the most relevant chunks related to my question. This helps confirm that the retrieval process is working correctly and that Gemini will receive only the most useful information when generating an answer.

In [ ]:
# Test retrieval before asking Gemini

# Ask a question about the meeting transcript
test_question = "What decisions were made during the meeting?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: What decisions were made during the meeting?
Retrieved 2 relevant chunks:

CHUNK 1:
BOARDROOM COMPANY STAFF MEETING
 TRANSCRIPT
Company: Apex Innovations Ltd.
Date: 15 July 2026
Time: 9:00 AM – 10:30 AM
Venue: Executive Boardroom
Chairperson: Mrs. Grace Okafor (Managing Director)
Attendees: Grace Okafor, Daniel Musa, Sarah James, Michael Adeyemi, Linda Eze, Peter
Johnson, Esther Bello.
Agenda
Review previous minutes; Quarterly sales; Budget; CRM upgrade; Customer feedback; Staff
training; Action items.
Discussion
The Managing Director welcomed everyone and reviewed progress since the last meeting. Sales
increased by 18% over the previous quarter. Finance approved additional funding for digital
transformation. The IT Manager confirmed that the new CRM system is 85% complete and security

-------------------------------------------------------
CHUNK 2:
transformation. The IT Manager confirmed that the new CRM system is 85% complete and security
testing will finish next week. HR

In [ ]:
# Test retrieval before asking Gemini

# Ask a question about the meeting transcript
test_question = "When is the application launch date?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: When is the application launch date?
Retrieved 2 relevant chunks:

CHUNK 1:
transformation. The IT Manager confirmed that the new CRM system is 85% complete and security
testing will finish next week. HR proposed a two-day staff training programme covering customer
service, cybersecurity, and communication. Customer Service recommended an automated
ticketing system to improve response times.
Decisions
Approve additional IT budget; Launch CRM on 30 July 2026; Conduct staff training on 22–23 July
2026; Improve customer response time to under two hours.
Action Items
Finance: Release IT budget. IT: Complete CRM testing. HR: Organize training. Sales: Prepare Q3
strategy. Customer Service: Deploy ticketing system. Managing Director: Review progress at the
next meeting.
Next Meeting
29 July 2026 at 9:00 AM in the Executive Boardroom.

-------------------------------------------------------
CHUNK 2:
BOARDROOM COMPANY STAFF MEETING
 TRANSCRIPT
Company: Apex Innovations Ltd.
Date: 15 J

In [ ]:
# Test retrieval before asking Gemini

# Ask a question about the meeting transcript
test_question = "What action items were assigned??"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: What action items were assigned??
Retrieved 2 relevant chunks:

CHUNK 1:
transformation. The IT Manager confirmed that the new CRM system is 85% complete and security
testing will finish next week. HR proposed a two-day staff training programme covering customer
service, cybersecurity, and communication. Customer Service recommended an automated
ticketing system to improve response times.
Decisions
Approve additional IT budget; Launch CRM on 30 July 2026; Conduct staff training on 22–23 July
2026; Improve customer response time to under two hours.
Action Items
Finance: Release IT budget. IT: Complete CRM testing. HR: Organize training. Sales: Prepare Q3
strategy. Customer Service: Deploy ticketing system. Managing Director: Review progress at the
next meeting.
Next Meeting
29 July 2026 at 9:00 AM in the Executive Boardroom.

-------------------------------------------------------
CHUNK 2:
BOARDROOM COMPANY STAFF MEETING
 TRANSCRIPT
Company: Apex Innovations Ltd.
Date: 15 July

**Changing it to something more specific**

In [ ]:
# Change this to something specific in YOUR meeting transcript
test_question = "What decisions were made during the meeting?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # Return top 3 chunks
retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: What decisions were made during the meeting?
Retrieved 2 relevant chunks:

CHUNK 1:
BOARDROOM COMPANY STAFF MEETING
 TRANSCRIPT
Company: Apex Innovations Ltd.
Date: 15 July 2026
Time: 9:00 AM – 10:30 AM
Venue: Executive Boardroom
Chairperson: Mrs. Grace Okafor (Managing Director)
Attendees: Grace Okafor, Daniel Musa, Sarah James, Michael Adeyemi, Linda Eze, Peter
Johnson, Esther Bello.
Agenda
Review previous minutes; Quarterly sales; Budget; CRM upgrade; Customer feedback; Staff
training; Action items.
Discussion
The Managing Director welcomed everyone and reviewed progress since the last meeting. Sales
increased by 18% over the previous quarter. Finance approved additional funding for digital
transformation. The IT Manager confirmed that the new CRM system is 85% complete and security

-------------------------------------------------------
CHUNK 2:
transformation. The IT Manager confirmed that the new CRM system is 85% complete and security
testing will finish next week. HR

**Step 8  Build the RAG Question Answering Function**

In this step, I connected the retrieval system to the Gemini Large Language Model. The function first searches ChromaDB for the three most relevant chunks from the meeting transcript, combines them into a single context, and sends that context with the user's question to Gemini. This ensures that the AI answers only from the uploaded document instead of using outside knowledge, making the responses more accurate and reliable.

In [ ]:
def ask_document(question, show_chunks=True):
    """Ask a question about the document using RAG + Gemini."""

    # Step 1: Retrieve relevant chunks
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5}) # Updated k to 5
    retrieved_chunks = retriever.invoke(question)

    # Step 2: Build the context string from retrieved chunks
    # Deduplicate chunks to avoid sending redundant information
    unique_chunks_content = set()
    deduplicated_chunks = []
    for chunk in retrieved_chunks:
        if chunk.page_content not in unique_chunks_content:
            unique_chunks_content.add(chunk.page_content)
            deduplicated_chunks.append(chunk)

    context = "\n\n---\n\n".join([chunk.page_content for chunk in deduplicated_chunks])

    # Step 3: Build the prompt
    user_prompt = f"""You are an AI assistant designed to extract and summarize information from meeting transcripts.
Based on the following CONTEXT from the meeting, please answer the QUESTION.
Your answer should be as comprehensive and informative as possible, drawing all relevant details and making logical inferences *only* from the provided CONTEXT.
If the CONTEXT does not contain enough information to provide a direct answer, or if the information is entirely absent, explain what information is missing or state that the answer cannot be found within the document.
Do not use any external knowledge.

CONTEXT:
{context}

QUESTION:
{question}
"""

    # Step 4: Ask Gemini
    response = llm.invoke(user_prompt)

    # Get Gemini's answer
    answer = response.content[0]["text"]

    # Print results
    print("=" * 65)
    print(f"QUESTION: {question}")
    print("=" * 65)
    print()
    print("ANSWER (from Gemini):")
    print(answer)

    # Show source chunks
    if show_chunks:
        print()
        print("SOURCE CHUNKS USED (what Gemini actually saw):")
        print("-" * 65)

        for i, chunk in enumerate(deduplicated_chunks): # Use deduplicated_chunks for display
            print(f"Chunk {i+1}:")
            print(chunk.page_content) # Display full content
            print()

    return answer


print("ask_document() function ready.")
print("Pipeline: Question → ChromaDB retrieval → Gemini → Answer")

ask_document() function ready.
Pipeline: Question → ChromaDB retrieval → Gemini → Answer


**Step 9  Test the RAG AI Agent**

In this step, I tested the complete RAG pipeline to confirm that the system works correctly. The uploaded meeting transcript is searched using ChromaDB, the most relevant chunks are retrieved, and Gemini uses only those retrieved sections to generate an accurate summary. This verifies that the AI agent can answer questions based on the document rather than relying on general knowledge.

In [ ]:
ask_document("Summarize this meeting.")

QUESTION: Summarize this meeting.

ANSWER (from Gemini):
Based on the provided transcript, here is a comprehensive summary of the Apex Innovations Ltd. staff meeting:

### **Meeting Overview**
* **Company:** Apex Innovations Ltd.
* **Date & Time:** 15 July 2026, 9:00 AM – 10:30 AM
* **Venue:** Executive Boardroom
* **Chairperson:** Mrs. Grace Okafor (Managing Director)
* **Attendees:** Grace Okafor, Daniel Musa, Sarah James, Michael Adeyemi, Linda Eze, Peter Johnson, and Esther Bello.

---

### **Key Discussions**
* **Sales Performance:** Sales increased by 18% compared to the previous quarter.
* **Finance & Digital Transformation:** Finance approved additional funding to support digital transformation.
* **CRM Upgrade:** The IT Manager reported that the new CRM system is 85% complete, with security testing scheduled to finish the following week.
* **Staff Training:** HR proposed a two-day training program for staff, focusing on customer service, cybersecurity, and communication.
* **C

'Based on the provided transcript, here is a comprehensive summary of the Apex Innovations Ltd. staff meeting:\n\n### **Meeting Overview**\n* **Company:** Apex Innovations Ltd.\n* **Date & Time:** 15 July 2026, 9:00 AM – 10:30 AM\n* **Venue:** Executive Boardroom\n* **Chairperson:** Mrs. Grace Okafor (Managing Director)\n* **Attendees:** Grace Okafor, Daniel Musa, Sarah James, Michael Adeyemi, Linda Eze, Peter Johnson, and Esther Bello.\n\n---\n\n### **Key Discussions**\n* **Sales Performance:** Sales increased by 18% compared to the previous quarter.\n* **Finance & Digital Transformation:** Finance approved additional funding to support digital transformation.\n* **CRM Upgrade:** The IT Manager reported that the new CRM system is 85% complete, with security testing scheduled to finish the following week.\n* **Staff Training:** HR proposed a two-day training program for staff, focusing on customer service, cybersecurity, and communication.\n* **Customer Service:** To improve response t

In [ ]:
ask_document("Who attended the meeting?")

QUESTION: Who attended the meeting?

ANSWER (from Gemini):
Based on the provided meeting transcript, the following individuals attended the meeting:

*   **John Williams** – Project Manager
*   **Sarah Johnson** – UI/UX Designer
*   **David Brown** – Software Developer
*   **Mary Adams** – Marketing Lead
*   **Michael Green** – Quality Assurance Engineer
*   **Linda Thomas** – Customer Support Manager

SOURCE CHUNKS USED (what Gemini actually saw):
-----------------------------------------------------------------
Chunk 1:
 David: Finish payment integration, fix bugs, complete API security review.  
 Michael: Investigate login issue, perform security testing, organize final system testing.  
 Mary: Launch marketing campaign, prepare budget proposal, organize product 
demonstration.  
 Linda: Complete user documentation, FAQs, tutorial videos, and customer support 
training materials.  
 John: Monitor project progress and approve final launch activities.  
 
Closing Remarks 
John: T

'Based on the provided meeting transcript, the following individuals attended the meeting:\n\n*   **John Williams** – Project Manager\n*   **Sarah Johnson** – UI/UX Designer\n*   **David Brown** – Software Developer\n*   **Mary Adams** – Marketing Lead\n*   **Michael Green** – Quality Assurance Engineer\n*   **Linda Thomas** – Customer Support Manager'

In [ ]:
ask_document("What tasks were assigned to each team member?")

QUESTION: What tasks were assigned to each team member?

ANSWER (from Gemini):
Based on the provided meeting transcript, tasks (Action Items) were assigned to specific departments and roles rather than to most individual team members by name. 

The only team member whose individual task can be directly identified is:
*   **Mrs. Grace Okafor (Managing Director):** Assigned to review progress at the next meeting.

The remaining tasks were assigned to the following departments/roles:
*   **Finance:** Release the IT budget.
*   **IT:** Complete CRM testing.
*   **HR:** Organize the staff training.
*   **Sales:** Prepare the Q3 strategy.
*   **Customer Service:** Deploy the automated ticketing system.

**Missing Information:**
While the transcript lists the other meeting attendees (**Daniel Musa, Sarah James, Michael Adeyemi, Linda Eze, Peter Johnson, and Esther Bello**), it does not specify which departments they represent or which individual roles they hold. Therefore, it is not possible 

'Based on the provided meeting transcript, tasks (Action Items) were assigned to specific departments and roles rather than to most individual team members by name. \n\nThe only team member whose individual task can be directly identified is:\n*   **Mrs. Grace Okafor (Managing Director):** Assigned to review progress at the next meeting.\n\nThe remaining tasks were assigned to the following departments/roles:\n*   **Finance:** Release the IT budget.\n*   **IT:** Complete CRM testing.\n*   **HR:** Organize the staff training.\n*   **Sales:** Prepare the Q3 strategy.\n*   **Customer Service:** Deploy the automated ticketing system.\n\n**Missing Information:**\nWhile the transcript lists the other meeting attendees (**Daniel Musa, Sarah James, Michael Adeyemi, Linda Eze, Peter Johnson, and Esther Bello**), it does not specify which departments they represent or which individual roles they hold. Therefore, it is not possible to determine which specific tasks were assigned to each of these 

## Test the updated `ask_document` function with a new question

In [ ]:
ask_document("What are the key details regarding the product launch?")

QUESTION: What are the key details regarding the product launch?

ANSWER (from Gemini):
Based on the provided meeting transcript, there is no explicit mention of a "product launch." However, the text contains detailed information regarding the launch and implementation of the company's new **CRM (Customer Relationship Management) system**, which appears to be the primary system launch discussed. 

The key details regarding the CRM launch are:

*   **Launch Date:** The CRM is scheduled to launch on **30 July 2026**.
*   **Current Progress:** The IT Manager confirmed that the new CRM system is currently **85% complete**.
*   **Testing:** Security testing for the system is scheduled to finish "next week" (relative to the meeting date of 15 July 2026). Completing the CRM testing is an official action item assigned to the IT department.
*   **Staff Training:** To prepare for the rollout, HR will organize a two-day staff training program on **22–23 July 2026** (prior to the launch), covering

'Based on the provided meeting transcript, there is no explicit mention of a "product launch." However, the text contains detailed information regarding the launch and implementation of the company\'s new **CRM (Customer Relationship Management) system**, which appears to be the primary system launch discussed. \n\nThe key details regarding the CRM launch are:\n\n*   **Launch Date:** The CRM is scheduled to launch on **30 July 2026**.\n*   **Current Progress:** The IT Manager confirmed that the new CRM system is currently **85% complete**.\n*   **Testing:** Security testing for the system is scheduled to finish "next week" (relative to the meeting date of 15 July 2026). Completing the CRM testing is an official action item assigned to the IT department.\n*   **Staff Training:** To prepare for the rollout, HR will organize a two-day staff training program on **22–23 July 2026** (prior to the launch), covering customer service, cybersecurity, and communication.\n*   **Funding:** Addition

## Creating `app.py` for Streamlit Deployment


**Step 10  Build and Deploy the Streamlit Web Application**

In this step, I converted my RAG system into an interactive Streamlit web application. The app allows users to upload a meeting transcript in PDF format, automatically processes and stores it in ChromaDB, and lets users ask questions about the meeting. Gemini then uses the retrieved document chunks to generate accurate answers, making the AI agent easy to use through a simple web interface.

In [7]:
%%writefile app.py

import streamlit as st
import os
from datetime import datetime # Import datetime

# LangChain components
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings # Updated import
from langchain_community.vectorstores import Chroma

# Gr
from langchain_groq import ChatGroq

@st.cache_resource
def load_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

# Authentication and DB
from auth_db import init_db, register_user, verify_user, save_conversation, get_conversations

# --- Streamlit UI Configuration ---
st.set_page_config(
    page_title="MeetingMind AI Agent",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.image("meetingmind_banner.png", use_container_width=True)

# Initialize the database
init_db()

# Session state for authentication
if 'authenticated' not in st.session_state:
    st.session_state.authenticated = False
if 'user_id' not in st.session_state:
    st.session_state.user_id = None
if 'username' not in st.session_state:
    st.session_state.username = None


def show_auth_page():
    """Displays the login/signup forms."""
    st.markdown("""
    Welcome! Please Login or Sign Up to use the MeetingMind AI Agent.
    """)

    tab1, tab2 = st.tabs(["Login", "Sign Up"])

    with tab1:
        st.subheader("Login") # Changed st.text back to st.subheader
        login_username = st.text_input("Username", key="login_username")
        login_password = st.text_input("Password", type="password", key="login_password")
        if st.button("Login"):
            user_id = verify_user(login_username, login_password)
            if user_id:
                st.session_state.authenticated = True
                st.session_state.user_id = user_id
                st.session_state.username = login_username
                st.rerun()
            else:
                st.error("Invalid username or password")

    with tab2:
        st.subheader("Sign Up")
        new_username = st.text_input("New Username", key="new_username")
        new_password = st.text_input("New Password", type="password", key="new_password")
        confirm_password = st.text_input("Confirm Password", type="password", key="confirm_password")
        if st.button("Register"):
            if new_password != confirm_password:
                st.error("Passwords do not match!")
            elif len(new_username) < 3 or len(new_password) < 6:
                st.error("Username must be at least 3 characters and password at least 6 characters long.")
            else:
                if register_user(new_username, new_password):
                    st.success("Registration successful! Please login.")
                else:
                    st.error("Username already exists or registration failed.")


def logout():
    st.session_state.authenticated = False
    st.session_state.user_id = None
    st.session_state.username = None
    st.session_state.vectorstore = None
    st.session_state.pdf_uploaded = False
    st.rerun()


if not st.session_state.authenticated:
    show_auth_page()
else:
    with st.sidebar:
        st.header(f"Welcome, {st.session_state.username}!")
        if st.button("Logout"):
            logout()
        st.markdown("--- ")
        st.header("📚 How to Use")
        with st.expander("Click here for detailed instructions", expanded=True):
            st.markdown("""
            1.  **Upload PDF**: Use the file uploader below to select your meeting transcript in PDF format.
            2.  **Ask Questions**: Once processed, type your questions about the meeting content into the text area.
            3.  **Get Answers**: Click 'Get Answer' to receive a concise, accurate response generated by Google Gemini, based *only* on your uploaded document.
            """)
        st.markdown("--- ")
        st.header("💡 About This App")
        st.info("""
        MeetingMind AI Agent helps you cut through lengthy meeting notes to find key information fast. It uses advanced AI to summarize, extract decisions, and answer questions directly from your documents.

        Developed as a capstone project utilizing LangChain, ChromaDB, HuggingFace Embeddings, and Google Gemini.
        """)

        st.markdown("--- ")
        st.header("Past Conversations")
        user_conversations = get_conversations(st.session_state.user_id)
        if user_conversations:
            for i, (timestamp, question, answer) in enumerate(user_conversations):
                with st.expander(f"**{datetime.fromisoformat(timestamp).strftime('%Y-%m-%d %H:%M')}**: {question[:50]}..."):
                    st.markdown(f"**Q:** {question}")
                    st.markdown(f"**A:** {answer}")
        else:
            st.info("No past conversations found.")

    if 'vectorstore' not in st.session_state:
        st.session_state.vectorstore = None
    if 'pdf_uploaded' not in st.session_state:
        st.session_state.pdf_uploaded = False

    # Groq API Key input (using st.secrets for security)
    groq_api_key = st.secrets.get("GROQ_API_KEY", "")
    with st.container(border=True):
        st.subheader("Upload Your Meeting Transcript (PDF)")
        uploaded_file = st.file_uploader("Choose a PDF file to analyze", type="pdf", label_visibility="collapsed")

        if uploaded_file is not None:
            st.session_state.pdf_uploaded = True
            st.success("PDF uploaded successfully! Now processing...")

            with open("uploaded_meeting.pdf", "wb") as f:
                f.write(uploaded_file.getbuffer())
            doc_filename = "uploaded_meeting.pdf"

            st.write("#### Document Processing Status")

            with st.spinner("Loading PDF..."):
                try:
                    loader = PyPDFLoader(doc_filename)
                    documents = loader.load()
                    st.success(f"Loaded {len(documents)} pages.")
                except Exception as e:
                    st.error(f"Error loading PDF: {e}")
                    st.error("This could be due to a corrupted PDF, a password-protected file, or an unsupported format.")
                    st.error("Please ensure the PDF is valid and not encrypted.")
                    st.stop() # Stop further execution if PDF loading fails

            with st.spinner("Splitting document into chunks..."):
                text_splitter = RecursiveCharacterTextSplitter(
                    chunk_size=1200,
                    chunk_overlap=200
                )
                chunks = text_splitter.split_documents(documents)
                st.success(f"Created {len(chunks)} chunks.")

            with st.spinner("Embedding chunks and creating vector store (this may take a moment)..."):
                embeddings = load_embeddings()

                st.session_state.vectorstore = Chroma.from_documents(
                    documents=chunks,
                    embedding=embeddings
                )

                st.success("Vector store created successfully!")
    if st.session_state.pdf_uploaded and st.session_state.vectorstore is not None:
        st.markdown("--- ")
        with st.container(border=True):
            st.subheader("Ask a Question About the Meeting")
            question_input = st.text_area("Enter your question here:", key="question_area", height=100)

    if st.button("Get Answer"):

        if not groq_api_key:
            st.error("Groq API Key not found in Streamlit Secrets.")

        else:
            llm = ChatGroq(
                groq_api_key=groq_api_key,
                model_name="llama-3.3-70b-versatile",
                temperature=0
            )
            def ask_document(question):
                retriever = st.session_state.vectorstore.as_retriever(
                    search_kwargs={"k": 8}
                )

                retrieved_chunks = retriever.invoke(question)

                unique_chunks_content = set()
                deduplicated_chunks = []
                for chunk in retrieved_chunks:
                    if chunk.page_content not in unique_chunks_content:
                        unique_chunks_content.add(chunk.page_content)
                        deduplicated_chunks.append(chunk)

                context = "\n\n---\n\n".join(
                    [chunk.page_content for chunk in deduplicated_chunks]
                )

                user_prompt = f"""You are an AI assistant designed to extract and summarize information from meeting transcripts.\nBased on the following CONTEXT from the meeting, please answer the QUESTION.\nYour answer should be as comprehensive and informative as possible, drawing all relevant details and making logical inferences *only* from the provided CONTEXT.\nIf the CONTEXT does not contain enough information to provide a direct answer, or if the information is entirely absent, explain what information is missing or state that the answer cannot be found within the document.\nDo not use any external knowledge.\n\nCONTEXT:\n{context}\n\nQUESTION:\n{question}\n"""

                try:
                    response = llm.invoke(user_prompt)
                    answer = str(response.content)

                except Exception as e:
                    st.error(f"Groq Error: {e}")
                    return (
                        "⚠️ Groq is temporarily unavailable.",
                        []
                    )

                return answer, deduplicated_chunks
            with st.spinner("Generating answer with Groq AI..."):
                answer, source_chunks = ask_document(question_input)

                st.success("Answer Generated!")
                st.markdown("#### Answer:")
                st.write(answer)

                # Save conversation
                save_conversation(st.session_state.user_id, question_input, answer)

                # Removed the st.expander that displayed source chunks
    else:
        st.info("Please upload a PDF document to begin asking questions.")

Overwriting app.py


## Creating `requirements.txt`

**Step 11  Create the Project Requirements File**

In this step, I created a requirements.txt file that lists all the Python

In [18]:
!cat requirements.txt


streamlit
langchain
langchain-community
langchain-text-splitters
langchain-huggingface
langchain-groq  # Added new requirement for Groq
chromadb
pypdf
sentence-transformers
torch
transformers
bcrypt
torchvision


**DATABASE**

In [3]:
%%writefile auth_db.py

import sqlite3
import bcrypt
import json
from datetime import datetime

DATABASE_NAME = "users.db"

def init_db():
    """Initializes the SQLite database for users and conversations."""
    conn = sqlite3.connect(DATABASE_NAME)
    c = conn.cursor()

    # Create users table
    c.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL
        )
    """)

    # Create conversations table
    c.execute("""
        CREATE TABLE IF NOT EXISTS conversations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            timestamp TEXT NOT NULL,
            question TEXT NOT NULL,
            answer TEXT NOT NULL,
            FOREIGN KEY (user_id) REFERENCES users(id)
        )
    """)

    conn.commit()
    conn.close()
    print(f"Database '{DATABASE_NAME}' initialized.")

def register_user(username, password):
    """Registers a new user with a hashed password. Returns True on success, False if user exists."""
    conn = sqlite3.connect(DATABASE_NAME)
    c = conn.cursor()
    hashed_password = bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt())
    try:
        c.execute("INSERT INTO users (username, password) VALUES (?, ?)", (username, hashed_password.decode('utf-8')))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        # Username already exists
        return False
    finally:
        conn.close()

def verify_user(username, password):
    """Verifies user credentials. Returns user_id on success, None otherwise."""
    conn = sqlite3.connect(DATABASE_NAME)
    c = conn.cursor()
    c.execute("SELECT id, password FROM users WHERE username = ?", (username,))
    result = c.fetchone()
    conn.close()
    if result:
        user_id, hashed_password = result
        if bcrypt.checkpw(password.encode('utf-8'), hashed_password.encode('utf-8')):
            return user_id
    return None

def save_conversation(user_id, question, answer):
    """Saves a question and its answer to the conversation history for a given user."""
    conn = sqlite3.connect(DATABASE_NAME)
    c = conn.cursor()
    timestamp = datetime.now().isoformat()
    c.execute(
        "INSERT INTO conversations (user_id, timestamp, question, answer) VALUES (?, ?, ?, ?)",
        (user_id, timestamp, question, answer)
    )
    conn.commit()
    conn.close()

def get_conversations(user_id):
    """Retrieves all conversation history for a given user."""
    conn = sqlite3.connect(DATABASE_NAME)
    c = conn.cursor()
    c.execute("SELECT timestamp, question, answer FROM conversations WHERE user_id = ? ORDER BY timestamp DESC", (user_id,))
    conversations = c.fetchall()
    conn.close()
    return conversations

# Initialize the database when the module is imported (or on first run)
init_db()


Writing auth_db.py


**Step 12 – Create the Project Documentation (README)**

In this step, I created a README.md file to document the project. The README provides an overview of the application, its features, the technologies used, installation instructions, and usage information. It helps anyone who visits the GitHub repository understand the purpose of the project and how to run it successfully.

In [10]:
%%writefile README.md

# MeetingMind AI Agent

MeetingMind AI Agent is an intelligent meeting assistant that leverages Artificial Intelligence, specifically a Large Language Model (LLM) and Retrieval-Augmented Generation (RAG). The MeetingMind AI Agent is designed to simplify meeting management and enhance productivity. It does this by:

- Analyzing meeting transcripts and notes: It uses a Large Language Model (LLM) to process the content of uploaded PDF meeting documents.
- Generating structured outputs: It automatically creates executive summaries, highlights key decisions, identifies action items, drafts follow-up emails, and suggests agendas for future meetings. This helps users quickly grasp the essential outcomes of a meeting.
- Facilitating question answering with Retrieval-Augmented Generation (RAG): Users can ask specific questions about the uploaded meeting documents. The RAG system ensures that the answers are directly derived from the meeting content, providing accurate and contextually relevant information rather than relying on general knowledge.

In essence, MeetingMind aims to solve the problem of information overload from lengthy meeting notes and transcripts, making it easier to extract crucial details like decisions, responsibilities, deadlines, and follow-up actions, which might otherwise be overlooked.

## Features

- Upload meeting transcript PDFs
- Automatic text chunking
- Semantic search using ChromaDB
- Embeddings using all-MiniLM-L6-v2
- Question answering with Groq

## Technologies Used

- Python
- Streamlit
- LangChain
- Groq
- ChromaDB
- HuggingFace Embeddings
- langchain-huggingface
- PyPDF
- Sentence Transformers
- Torch
- Transformers

## Functionality of the Application
You can check the functionality of this model by clicking the drive's link below:
https://drive.google.com/file/d/1BPN5zkt1VYyQqy0_sqOuYh68HtpY2sU5/view?usp=sharing

## Link to the Application
Here is a link to the Web Application

https://meetingmind-ai-agent-a-retrieval-augmented-meeting-assistant-j.streamlit.app/


## Usage

To run the Streamlit application locally, navigate to the project directory in your terminal and execute the following command:

```bash
streamlit run app.py
```

Once the application is running, you can:

1.  **Upload a PDF:** Use the file uploader to select your meeting transcript in PDF format.
2.  **Ask Questions:** Type your questions into the provided text area and click 'Get Answer'.
3.  **Review Answers:** The application will display the answer generated by Groq.

## Installation

```bash
pip install -r requirements.txt

Overwriting README.md
